# PM 변수 포함/제외 모델 성능 비교 분석

## 목표
- PM 변수(PM10, PM25) **제외** 버전과 **포함** 버전의 성능 차이를 비교합니다.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import platform
import warnings
warnings.filterwarnings("ignore")

if platform.system() == "Windows":
    plt.rc("font", family="Malgun Gothic")
plt.rcParams["axes.unicode_minus"] = False

print("라이브러리 로드 완료!")


In [ ]:
train_x = pd.read_csv("train_x.csv")
train_y = pd.read_csv("train_y.csv")
test_x = pd.read_csv("test_x.csv")
test_y = pd.read_csv("test_y.csv")

def preprocess_base(df):
    df = df.copy()
    if "time" in df.columns:
        df["time"] = pd.to_datetime(df["time"])
        if "hour" not in df.columns:
            df["hour"] = df["time"].dt.hour
        df = df.drop(columns=["time"])
    return df

if "time" in train_y.columns:
    train_y = train_y.drop(columns=["time"])
if "time" in test_y.columns:
    test_y = test_y.drop(columns=["time"])

def preprocess_no_pm(df):
    df = preprocess_base(df)
    pm_cols = [col for col in df.columns if "PM" in col]
    df = df.drop(columns=pm_cols)
    return df

def preprocess_with_pm(df):
    return preprocess_base(df)

train_x_no_pm = preprocess_no_pm(train_x)
test_x_no_pm = preprocess_no_pm(test_x)
train_x_with_pm = preprocess_with_pm(train_x)
test_x_with_pm = preprocess_with_pm(test_x)

common_no_pm = sorted(list(set(train_x_no_pm.columns) & set(test_x_no_pm.columns)))
train_x_no_pm = train_x_no_pm[common_no_pm].ffill().bfill()
test_x_no_pm = test_x_no_pm[common_no_pm].ffill().bfill()

common_with_pm = sorted(list(set(train_x_with_pm.columns) & set(test_x_with_pm.columns)))
train_x_with_pm = train_x_with_pm[common_with_pm].ffill().bfill()
test_x_with_pm = test_x_with_pm[common_with_pm].ffill().bfill()

print(f"PM 제외 변수 수: {len(common_no_pm)}개")
print(f"PM 포함 변수 수: {len(common_with_pm)}개")
print(f"PM 관련 추가 변수: {sorted(set(common_with_pm) - set(common_no_pm))}")


In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "SVR": SVR(kernel="rbf", C=10, gamma=0.1),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42, verbosity=0),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42)
}

def evaluate_models(train_x, test_x, train_y, test_y, label):
    scaler = StandardScaler()
    train_x_scaled = scaler.fit_transform(train_x)
    test_x_scaled = scaler.transform(test_x)
    results = []
    for name, model in models.items():
        print(f"  [{label}] {name} 학습 중...")
        model.fit(train_x_scaled, train_y.values.ravel())
        pred_y = model.predict(test_x_scaled)
        r2 = r2_score(test_y, pred_y)
        mae = mean_absolute_error(test_y, pred_y)
        rmse = np.sqrt(mean_squared_error(test_y, pred_y))
        results.append({"Model": name, "R2": r2, "MAE": mae, "RMSE": rmse, "버전": label})
    return pd.DataFrame(results)

print("=== PM 제외 버전 학습 시작 ===")
df_no_pm = evaluate_models(train_x_no_pm, test_x_no_pm, train_y, test_y, "PM 제외")

print("\n=== PM 포함 버전 학습 시작 ===")
df_with_pm = evaluate_models(train_x_with_pm, test_x_with_pm, train_y, test_y, "PM 포함")

print("\n학습 완료!")


In [ ]:
print("=" * 70)
print("PM 제외 버전 결과 (R2 내림차순)")
print("=" * 70)
display(df_no_pm.sort_values("R2", ascending=False).reset_index(drop=True))

print("\n" + "=" * 70)
print("PM 포함 버전 결과 (R2 내림차순)")
print("=" * 70)
display(df_with_pm.sort_values("R2", ascending=False).reset_index(drop=True))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("PM 변수 포함/제외 모델 성능 비교", fontsize=16, fontweight="bold")

metrics = ["R2", "MAE", "RMSE"]
titles = ["R2 Score (높을수록 좋음)", "MAE (낮을수록 좋음)", "RMSE (낮을수록 좋음)"]
colors = ["#2196F3", "#FF5722"]

for idx, (metric, title) in enumerate(zip(metrics, titles)):
    ax = axes[idx]
    model_names_sorted = df_no_pm.sort_values("Model")["Model"].values
    x = np.arange(len(model_names_sorted))
    width = 0.35
    vals_no_pm = df_no_pm.sort_values("Model")[metric].values
    vals_with_pm = df_with_pm.sort_values("Model")[metric].values
    ax.bar(x - width/2, vals_no_pm, width, label="PM 제외", color=colors[0], alpha=0.8)
    ax.bar(x + width/2, vals_with_pm, width, label="PM 포함", color=colors[1], alpha=0.8)
    ax.set_xlabel("모델", fontsize=11)
    ax.set_ylabel(metric, fontsize=11)
    ax.set_title(title, fontsize=12)
    ax.set_xticks(x)
    ax.set_xticklabels(model_names_sorted, rotation=30, ha="right", fontsize=9)
    ax.legend()
    ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("PM_비교_분석_결과.png", dpi=150, bbox_inches="tight")
plt.show()
print("차트 저장 완료: PM_비교_분석_결과.png")


In [ ]:
print("=" * 70)
print("PM 포함 vs PM 제외: 성능 차이 분석")
print("=" * 70)

diff_data = []
for model_name in df_no_pm["Model"].values:
    r2_no = df_no_pm[df_no_pm["Model"] == model_name]["R2"].values[0]
    r2_with = df_with_pm[df_with_pm["Model"] == model_name]["R2"].values[0]
    mae_no = df_no_pm[df_no_pm["Model"] == model_name]["MAE"].values[0]
    mae_with = df_with_pm[df_with_pm["Model"] == model_name]["MAE"].values[0]
    rmse_no = df_no_pm[df_no_pm["Model"] == model_name]["RMSE"].values[0]
    rmse_with = df_with_pm[df_with_pm["Model"] == model_name]["RMSE"].values[0]
    diff_data.append({
        "Model": model_name,
        "R2_PM제외": round(r2_no, 4),
        "R2_PM포함": round(r2_with, 4),
        "R2_차이(포함-제외)": round(r2_with - r2_no, 4),
        "MAE_PM제외": round(mae_no, 2),
        "MAE_PM포함": round(mae_with, 2),
        "MAE_차이": round(mae_with - mae_no, 2),
        "RMSE_PM제외": round(rmse_no, 2),
        "RMSE_PM포함": round(rmse_with, 2),
        "RMSE_차이": round(rmse_with - rmse_no, 2)
    })

df_diff = pd.DataFrame(diff_data)
display(df_diff)

avg_r2_diff = df_diff["R2_차이(포함-제외)"].mean()
print(f"\n평균 R2 향상: {avg_r2_diff:.4f}")
print("양수 = PM 포함이 더 좋음, 음수 = PM 제외가 더 좋음")
